In [ ]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import StandardScaler
from sklearn.metrics.pairwise import cosine_similarity

from sentence_transformers import SentenceTransformer

In [22]:
df = pd.read_csv("/dataset.csv", encoding="utf-8")
print(df.shape)
df.head()

(17643, 21)


,Unnamed: 0,track_id,artists,album_name,track_name,popularity,duration_ms,explicit,danceability,energy,...,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature,track_genre
0,0,5SuOikwiRyPMVoIQDJUgSV,Gen Hoshino,Comedy,Comedy,73.0,230666.0,False,0.676,0.4610,...,-6.746,0.0,0.1430,0.0322,0.000001,0.3580,0.715,87.917,4.0,acoustic
1,1,4qPNDBW1i3p13qLCt0Ki3A,Ben Woodward,Ghost (Acoustic),Ghost - Acoustic,55.0,149610.0,False,0.420,0.1660,...,-17.235,1.0,0.0763,0.9240,0.000006,0.1010,0.267,77.489,4.0,acoustic
2,2,1iJBSr7s7jYXzM8EGcbK5b,Ingrid Michaelson;ZAYN,To Begin Again,To Begin Again,57.0,210826.0,False,0.438,0.3590,...,-9.734,1.0,0.0557,0.2100,0.000000,0.1170,0.120,76.332,4.0,acoustic
3,3,6lfxq3CG4xtTiEg7opyCyx,Kina Grannis,Crazy Rich Asians (Original Motion Picture Sou...,Can't Help Falling In Love,71.0,201933.0,False,0.266,0.0596,...,-18.515,1.0,0.0363,0.9050,0.000071,0.1320,0.143,181.740,3.0,acoustic
4,4,5vjLSffimiIP26QG5WcN2K,Chord Overstreet,Hold On,Hold On,82.0,198853.0,False,0.618,0.4430,...,-9.681,1.0,0.0526,0.4690,0.000000,0.0829,0.167,119.949,4.0,acoustic


In [23]:
df.drop_duplicates(subset="track_name", inplace = True)

df = df.dropna(subset=[
    'danceability',
    'energy',
    'loudness',
'speechiness',
'acousticness',
'instrumentalness',
'liveness',
'valence',
'tempo'])

In [24]:
audio_features = [
    'danceability',
    'energy',
    'loudness',
'speechiness',
'acousticness',
'instrumentalness',
'liveness',
'valence',
'tempo']

In [25]:
scaler = StandardScaler()

scaled_audio = scaler.fit_transform(df[audio_features])

In [26]:
audio_similarity = cosine_similarity(scaled_audio)

In [27]:
genre_dummies = pd.get_dummies(df['track_genre'])

genre_similarity = cosine_similarity(genre_dummies)

In [28]:
df['popularity_norm'] = df['popularity'] / df['popularity'].max()

In [29]:
df['text_feature'] = (
    df['track_name'] + " " +
    df['artists'] + " " +
    df['album_name'] + " " +
    df['track_genre']
)

In [30]:
model = SentenceTransformer("all-MiniLM-L6-v2")

text_embeddings = model.encode(df['text_feature'].tolist(),
                              show_progress_bar = True)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/457 [00:00<?, ?it/s]

In [31]:
nlp_similarity = cosine_similarity(text_embeddings)

In [32]:
HYBRID_AUDIO_WEIGHT = 0.5
HYBRID_GENRE_WEIGHT = 0.2
HYBRID_NLP_WEIGHT = 0.3

hybrid_similarity = (
    HYBRID_AUDIO_WEIGHT * audio_similarity +
    HYBRID_GENRE_WEIGHT * genre_similarity +
    HYBRID_NLP_WEIGHT * nlp_similarity
)

In [35]:
from sentence_transformers.util import similarity
def recommend(song_name, top_n=10):

  song_index = df[df['track_name'] == song_name].index

  if len(song_index) == 0:
    print('Song not found')
    return
  song_index = song_index[0]

  similarity_scores = list(enumerate(hybrid_similarity[song_index]))

  similarity_scores = sorted(
      similarity_scores,
      key = lambda x: x[1],
      reverse = True
  )
  similarity_scores = similarity_scores[1:top_n+1]

  song_indices = [i[0] for i in similarity_scores]

  recommendations = df.iloc[song_indices][[
      'track_name',
      'artists',
      'track_genre',
      'popularity'
  ]]
  return recommendations

In [38]:
recommend('To Begin Again')

,track_name,artists,track_genre,popularity
67,She Used To Be Mine,Sara Bareilles,acoustic,67.0
461,Start of Time,Gabrielle Aplin,acoustic,45.0
700,How Far Does the Dark Go?,Anya Marina,acoustic,32.0
176,The Story,Brandi Carlile,acoustic,66.0
575,Such A Simple Thing - Recorded at Sound Stage ...,Ray LaMontagne,acoustic,56.0
918,Over You,Ingrid Michaelson;A Great Big World,acoustic,47.0
71,Superman (It's Not Easy),Five For Fighting,acoustic,70.0
544,Linda Ronstadt,AJJ,acoustic,31.0
356,I Won't Let You Down,Erin McCarley,acoustic,40.0
477,Time After Time,Boyce Avenue;Megan Davies;Jaclyn Davies,acoustic,60.0


In [39]:
def recommend_by_features(user_features, top_n = 10):

  user_array = scaler.transform([user_features])

  similarity = cosine_similarity(user_array, scaled_audio)

  top_indices = similarity[0].argsort()[::-1][:top_n]

  return df.iloc[top_indices][[
      'track_name',
      'artists',
      'track_genre'
  ]]

In [40]:
recommend_by_features([
    0.8,  # danceability
    0.9,  # energy
    -5,   # loudness
    0.05,
    0.1,
    0.0,
    0.2,
    0.7,
    120
])

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


,track_name,artists,track_genre
14755,Colors Song for Children,ChuChu TV,children
1692,The Plan,The Poets Of Rhythm,afrobeat
1639,Novos Mundos,Afrocidade,afrobeat
10088,What's Going On?,Mekon;Roxanne Shante,breakbeat
1474,Pas la peine,Vaudou Game,afrobeat
13058,I Don't Wanna Go,Sterling Void,chicago-house
8561,You Do You,Zayde Wølf,blues
5091,Go Big or Go Home,ENHYPEN,anime
10319,Good Times,Krafty Kuts;Sporty-O,breakbeat
1524,Topo do Mundo,Afrocidade,afrobeat


In [41]:
import pickle

pickle.dump(scaler, open('scaler.pkl','wb'))
pickle.dump(hybrid_similarity, open('similarity.pkl','wb'))
pickle.dump(df, open('songs.pkl','wb'))

In [43]:
!pip install streamlit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 62.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 111.9 MB/s eta 0:00:00


In [45]:
import streamlit as st
import pickle

df = pickle.load(open('songs.pkl','rb'))

def recommend(song):

  index = df[df['track_name']==song].index[0]

  scores = list(enumerate(similarity[index]))

  scores = sorted(scores, key=lambda x:x[1], reverse=True)

  songs = [df.iloc[i[0]].track_name for i in scores[1:6]]

  return songs

st.title('Spotify Recommendation System')

song_list = df['track_name'].values

selected_song = st.selectbox("Select a song", song_list)

if st.button('Recommend'):
  recs = recommend(selected_song)

  for son in recs:
    st.write(song)

2026-03-09 05:47:38.794 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-09 05:47:38.798 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-09 05:47:38.799 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-09 05:47:38.808 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-09 05:47:38.812 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-09 05:47:38.830 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-09 05:47:38.846 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-09 05:47:38.849 Session state does not function when running a script without `streamlit run`
2026-03-09 05:47

In [49]:
!ls

sample_data  scaler.pkl  similarity.pkl  songs.pkl  spotify_data.csv


In [51]:
!pip install streamlit pyngrok

import subprocess
import time
from pyngrok import ngrok

# Add your token
ngrok.set_auth_token("3AhDtW3PL934wJseqTun6cv2s6c_2fZb7XTegfub4mzwUVNCe")

# Start streamlit
process = subprocess.Popen(['streamlit', 'run', 'app.py'])

time.sleep(5)

# Open tunnel
public_url = ngrok.connect(8501)

print("App URL:", public_url)

App URL: NgrokTunnel: "https://exoterically-pseudospectral-alma.ngrok-free.dev" -> "http://localhost:8501"
